<a href="https://colab.research.google.com/github/Marti6cp/BANA7075_ProjectTeam6/blob/main/BANA7075_FinalProject_Team6_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install great_expectations
!pip install dvc dvc-gdrive
!pip install mlflow
!pip install fastapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5

In [2]:
#####GitHub Configuration to Maintain Versioning --Martinez

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git config --global user.email "marti6cp@mail.uc.edu"
!git config --global user.name "Chris Martinez"

username = "marti6cp"
repo = "BANA7075_ProjectTeam6"

!git clone https://{token}@github.com/{username}/{repo}.git

%cd /content/BANA7075_ProjectTeam6




Cloning into 'BANA7075_ProjectTeam6'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 63 (delta 23), reused 63 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 112.48 KiB | 28.12 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/BANA7075_ProjectTeam6


In [ ]:
####DVC initialization ---Martinez

# !dvc init
# !git add .dvc .dvcignore
# !git commit -m "Initialize DVC"

In [3]:
#####KaggleHub DataIngest Module- Martinez

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load a DataFrame with a specific version of a CSV
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sharmajicoder/amazon-e-commerce", "amazon_ecommerce_1M.csv",
)
#print(df.head())


####DataProcessing

df.to_csv('rawData.csv', index=False)



## Drop Null frames and establish processed data frame
df_processed = df.dropna()


#drop duplicates--- purchase date does not containenough granularity to avoid
#removing real data especially in a large ecommerce dataset drop duplicate
#functionality commented below to be turned on for real-world datasets that
#have more precise datetime information.

# df_processed.drop_duplicates(inplace=True)
# df_processed.drop_duplicates(inplace=True)

##data type consistency
##Text/string
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    df_processed[col] = df_processed[col].astype("string")
    df_processed[col] = df_processed[col].str.lower()


##numerics
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",
                  ]:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

## Dates
df_processed["purchase_date"] = pd.to_datetime(df_processed["purchase_date"])

print(df_processed.dtypes)



100%|██████████| 127M/127M [00:00<00:00, 148MB/s]


user_id               string[python]
product_id            string[python]
category              string[python]
subcategory           string[python]
brand                 string[python]
price                        float64
discount                     float64
final_price                  float64
rating                       float64
review_count                   int64
stock                          int64
seller_id             string[python]
seller_rating                float64
purchase_date         datetime64[ns]
shipping_time_days             int64
location              string[python]
device                string[python]
payment_method        string[python]
is_returned                     bool
delivery_status       string[python]
dtype: object


In [ ]:
#Staging Data Files to be verisoned in S3 bucket
df_processed.to_csv("/content/BANA7075_ProjectTeam6/processedData.csv", index=False)


# 3. Tell DVC to track the newly saved file
!dvc add /content/BANA7075_ProjectTeam6/rawData.csv
!dvc add /content/BANA7075_ProjectTeam6/processedData.csv



# 4. Commit the DVC pointer metadata to Git
!git add /content/BANA7075_ProjectTeam6/rawData.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore
!git add /content/BANA7075_ProjectTeam6/processedData.csv.dvc #/content/BANA7075_ProjectTeam6/.gitignore


# !git commit -am "Updated Pandas raw DataFrame,Train & test via DVC"

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding dataset.csv to cache:   0% 0/1 [00:00<?, ?file/s]
Adding dataset.csv to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
Adding dataset.csv to cache: 100% 1/1 [00:00<00:00,  6.07file/s{'info': ''}]
                                                                            
  0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]
100% 1/1 [00:00<00:00,  2.31files/s{'info': ''}]
Adding...: 100% 1/1 [00:01<00:00,  1.90s/file{'info': ''}]

To track the changes with git, run:

	git add dataset.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?f

In [ ]:
##### DVC google drive access blocked  fix via remote S3 bucket

!dvc remote add -d myremote s3://cm-education-test-bucket-201302613730-us-east-2-an/bana-storage


In [ ]:
!dvc remote list
!git add .dvc/config
!git commit -m "Configure DVC default remote storage"
import os
from google.colab import userdata

# Retrieve secrets from Colab
aws_key = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret = userdata.get('AWS_SECRET_ACCESS_KEY')

# Set them as environment variables for AWS authentication
os.environ['AWS_ACCESS_KEY_ID'] = aws_key
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret
!dvc push

myremote        
s3://cm-education-test-bucket-201302613730-us-east-2-an/bana-storage    
(default)
On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [4]:
#######DataValidation Module######## - Martinez

import great_expectations as gx

###Setup GX Environment
context = gx.get_context()

###Connect to Data
# Variables
datasource_name = "ecommerce_datasource"
data_asset_name = "ecommerce_df_asset"
batch_definition_name = "big_batch"
suite_name  = "ecommerce_exp_suite"

#
#creating datasource
datasource = context.data_sources.add_pandas(
    name= datasource_name
    )
data_asset = datasource.add_dataframe_asset(
    name= data_asset_name
    )

#Create Batch Definition/Batch
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_processed}

    )

# Create Suite
suite = gx.ExpectationSuite(name = suite_name)
suite = context.suites.add(suite)

#

#### Define Expectations

##VRule to validate All training records are present
RecordMatch = gx.expectations.ExpectTableRowCountToEqual(
    value = 1000000

)
#Add to suite
suite.add_expectation(RecordMatch)

##Rule to validate all Data Columnn are present and in order
ColumnMatch = gx.expectations.ExpectTableColumnsToMatchOrderedList(
    column_list=["user_id", "product_id", "category", "subcategory", "brand",
                 "price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_id", "seller_rating","purchase_date","shipping_time_days",
                 "location", "device" ,"payment_method","is_returned",
                 "delivery_status"  ]
)

#add to suite
suite.add_expectation(ColumnMatch)

##Rule to validate that trainging dataframe does not contain missing data
column_list = df.columns.tolist()
for columns in column_list:       #iterates through the columns in data to check for missing data
    NoNulls = gx.expectations.ExpectColumnValuesToNotBeNull(
    column=columns
)
#add to suite
suite.add_expectation(NoNulls)

## Rule to validate Data Set still has correct nuber of columns
Column_number_check = gx.expectations.ExpectTableColumnCountToEqual(
    value=20
)

#add to suite
suite.add_expectation(Column_number_check)

## Rule to validate column as strings dType
for col in ["user_id", "product_id", "category", "subcategory", "brand",
            "seller_id", "location", "device" ,"payment_method", "delivery_status"]:
    string_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= col,
    type_= "str"
    )
    #add to suite
    suite.add_expectation(string_check)



##. Rule to validate column values are numeric dType
for col in ["price" , "discount", "final_price", "rating", "review_count",
    "stock","seller_rating","shipping_time_days",]:
    num_check = gx.expectations.ExpectColumnValuesToBeInTypeList(
    column= col,
    type_list = ["float", "int"]
    )
    suite.add_expectation(num_check)
#Add to suite





## Rule to validate column only contains dates dType
date_check = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "purchase_date",
    type_= "datetime64"
    )

#add to suite
suite.add_expectation(date_check)



##Rule to validate "is_returned is bool data type
isBooleanDtype = gx.expectations.ExpectColumnValuesToBeOfType(
    column= "is_returned",
    type_= "bool"
)
#add to suite
suite.add_expectation(isBooleanDtype)


defination_name = "ecommerce validation"
validation_definition = gx.ValidationDefinition(
    data=batch_definition,
    suite = suite,
    name = defination_name
)

validation_results = validation_definition.run(batch_parameters={"dataframe": df_processed})


print(validation_results.statistics)

if validation_results.success:
  print("Data Passed Validation!")
else:
  print("Data Failed Validation !!!")
  print(validation_results.get_failed_validation_results())

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpc8m_bpyc' for ephemeral docs site


Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

{'evaluated_expectations': 24, 'successful_expectations': 24, 'unsuccessful_expectations': 0, 'success_percent': 100.0}
Data Passed Validation!


In [ ]:
#####Implement FastAPI#####
import runpy

if not validation_results.success:
    raise ValueError("Data validation failed. Modeling stopped.")

runpy.run_path("ModelOpsModule.ipynb", run_name="__main__")